# Trajectory evaluation

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import phoenix as px
from phoenix.evals import OpenAIModel
from phoenix.experiments import run_experiment, evaluate_experiment
from phoenix.experiments.types import Example
from phoenix.experiments.evaluators import create_evaluator
from phoenix.otel import register
import pandas as pd
from datetime import datetime
import os
import nest_asyncio
nest_asyncio.apply()

In [3]:
from utils import run_agent, get_phoenix_endpoint

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: evaluating-agent-path
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://s172-29-49-155p6006.lab-aws-production.deeplearning.ai/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



1. create dataset of test cases
2. upload it to phoenix
3. run the agent on the test cases.
<br>
**Phoenix experiment** is a structured setup used to evaluate an AI agent's performance. It consists of three core components:

Dataset: A collection of test cases or examples.

Task: The function executed on each example in the dataset.

Evaluator: A function that grades or scores the task's output. 

-> this is all done automatically by the run_experiment method.

In [4]:
#building the test dataset
convergence_questions = [
    "What was the average quantity sold per transaction?",
    "What is the mean number of items per sale?", 
    "Calculate the typical quantity per transaction",
    "What's the mean transaction size in terms of quantity?",
    "On average, how many items were purchased per transaction?",
    "What is the average basket size per sale?",
    "Calculate the mean number of products per purchase",
    "What's the typical number of units per order?",
    "What is the average number of products bought per purchase?",
    "Tell me the mean quantity of items in a typical transaction",
    "How many items does a customer buy on average per transaction?",
    "What's the usual number of units in each sale?",
    "What is the typical amount of products per transaction?",
    "Show the mean number of items customers purchase per visit",
    "What's the average quantity of units per shopping trip?",
    "How many products do customers typically buy in one transaction?",
    "What is the standard basket size in terms of quantity?"
]

convergence = pd.DataFrame({
    "question": convergence_questions
})

time = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
dataset = px.Client().upload_dataset(
    dataframe = convergence,
    dataset_name = f"convergence-{time}",
    input_keys = ["question"]
)

📤 Uploading dataset...
💾 Examples uploaded: https://s172-29-49-155p6006.lab-aws-production.deeplearning.ai/datasets/RGF0YXNldDoy/examples
🗄️ Dataset version ID: RGF0YXNldFZlcnNpb246Mg==


In [5]:
#formatting the step messages for readability 
def format_message_steps(messages):
    """
    convert a list of messages objects into a readable format that shows the steps taken
    
    Args: 
        messages(list): list of message object containing role, content, tool calls, etc.
        
    Returns:
        str: readable string that contains the steps taken.
    """
    
    steps = []
    for message in messages:
        if message.get("role") == "user":
            steps.append(f"user: {message.get('content')}")
        
        elif message.get("role") == "system":
            steps.append(f"system: {message.get('content')}")
        
        elif message.get("role") == "assistant":
            if message.get("tool calls"):
                for tool_call in message["tool_calls"]:
                    tool_name = tool_call["function"]["name"]
                    steps.append(f"tool call: {tool_name}")
            else:
                steps.append(f"assistant: {message.get('content')}")
        elif message.get("role") == "tool":
            steps.append(f"tool response: {message.get('content')}")
        
    return "\n".join(steps)
            

since we have multiple test cases, we need Example object to iterate over individual test cases and give the experiment script the individual test case which in this case is an individual question to run the experiment. 

In [6]:
#function to run the agent on the test cases
def run_agent_and_track_path(example: Example) -> str:
    messages = [{"role": "user", "content": example.input.get("question")}]
    result = run_agent(messages)
    return {"path_len": len(result), "messages": format_message_steps(result)}

In [7]:
#running the experiments
experiment = run_experiment(
    dataset,
    run_agent_and_track_path,
    experiment_name = "convergence eval",
    experiment_description = "evaluating the convergence of the agent"
)

🧪 Experiment started.
📺 View dataset experiments: https://s172-29-49-155p6006.lab-aws-production.deeplearning.ai/datasets/RGF0YXNldDoy/experiments
🔗 View this experiment: https://s172-29-49-155p6006.lab-aws-production.deeplearning.ai/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDox


running tasks |          | 0/17 (0.0%) | ⏳ 00:00<? | ?it/s

Running agent with messages: [{'role': 'user', 'content': 'What was the average quantity sold per transaction?'}]
Added system prompt to messages
Starting router
Received response with tool calls: True
Processing tool calls
Starting router
Received response with tool calls: False
No tool calls, returning final response
Running agent with messages: [{'role': 'user', 'content': 'What is the mean number of items per sale?'}]
Added system prompt to messages
Starting router
Received response with tool calls: True
Processing tool calls
Starting router
Received response with tool calls: False
No tool calls, returning final response
Running agent with messages: [{'role': 'user', 'content': 'Calculate the typical quantity per transaction'}]
Added system prompt to messages
Starting router
Received response with tool calls: True
Processing tool calls
Starting router
Received response with tool calls: False
No tool calls, returning final response
Running agent with messages: [{'role': 'user', 'con

In [8]:
experiment.as_dataframe()

,output,input,example_id
run_id,,,
RXhwZXJpbWVudFJ1bjox,"{'path_len': 5, 'messages': 'user: What was th...",{'question': 'What was the average quantity so...,RGF0YXNldEV4YW1wbGU6MTg=
RXhwZXJpbWVudFJ1bjoy,"{'path_len': 5, 'messages': 'user: What is the...",{'question': 'What is the mean number of items...,RGF0YXNldEV4YW1wbGU6MTk=
RXhwZXJpbWVudFJ1bjoz,"{'path_len': 5, 'messages': 'user: Calculate t...",{'question': 'Calculate the typical quantity p...,RGF0YXNldEV4YW1wbGU6MjA=
RXhwZXJpbWVudFJ1bjo0,"{'path_len': 5, 'messages': 'user: What's the ...",{'question': 'What's the mean transaction size...,RGF0YXNldEV4YW1wbGU6MjE=
RXhwZXJpbWVudFJ1bjo1,"{'path_len': 5, 'messages': 'user: On average,...","{'question': 'On average, how many items were ...",RGF0YXNldEV4YW1wbGU6MjI=
RXhwZXJpbWVudFJ1bjo2,"{'path_len': 5, 'messages': 'user: What is the...",{'question': 'What is the average basket size ...,RGF0YXNldEV4YW1wbGU6MjM=
RXhwZXJpbWVudFJ1bjo3,"{'path_len': 5, 'messages': 'user: Calculate t...",{'question': 'Calculate the mean number of pro...,RGF0YXNldEV4YW1wbGU6MjQ=
RXhwZXJpbWVudFJ1bjo4,"{'path_len': 5, 'messages': 'user: What's the ...",{'question': 'What's the typical number of uni...,RGF0YXNldEV4YW1wbGU6MjU=
RXhwZXJpbWVudFJ1bjo5,"{'path_len': 5, 'messages': 'user: What is the...",{'question': 'What is the average number of pr...,RGF0YXNldEV4YW1wbGU6MjY=


here we are counting everything like system messages, user messages, tool calls, etc to count optimal steps which is fine if its consistant across all the test cases.

In [14]:
outputs = experiment.as_dataframe()["output"].to_dict().values()

In [16]:
optimal_length = min(out.get("path_len") for out in outputs if out and out.get("path_len") is not None)
print("optimal path lenght is: ", optimal_length)

optimal path lenght is:  5


In [21]:
@create_evaluator(name="Convergence Eval", kind="CODE")
def evaluate_path_length(output: str) -> float:
    if output and output.get("path_len"):
        return optimal_length/float(output.get("path_len"))
    else:
        return 0

In [22]:
experiment = evaluate_experiment(experiment,
                            evaluators=[evaluate_path_length])

🧠 Evaluation started.


running experiment evaluations |          | 0/17 (0.0%) | ⏳ 00:00<? | ?it/s


🔗 View this experiment: https://s172-29-49-155p6006.lab-aws-production.deeplearning.ai/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDox

Experiment Summary (11/03/26 01:24 PM +0000)
--------------------------------------------
          evaluator   n  n_scores  avg_score
0  Convergence Eval  17        17        1.0

Experiment Summary (11/03/26 01:24 PM +0000)
--------------------------------------------
          evaluator   n  n_errors  \
0  Convergence Eval  17        17   

                                           top_error  
0  NameError("name 'optimal_path_length' is not d...  

Experiment Summary (11/03/26 01:23 PM +0000)
--------------------------------------------
          evaluator   n  n_scores  avg_score
0  Convergence Eval  17        17        0.0

Tasks Summary (11/03/26 01:06 PM +0000)
---------------------------------------
   n_examples  n_runs  n_errors
0          17      17         0
